# 03 Yer Seviyesi - Çatlak + Bina Hasarı Tespiti (Tek Oturum, Çoklu Mimari)

**Veri:** kullanıcının kendi Roboflow projeleri — `akash-roy-edivz/building-crack-detection-ys4zf` (çatlak) ve `ofgg/maras-tpx5p` (bina hasarı).  
**Bizim yöntem (production):** 2× YOLOv8 (catlak.pt + bina.pt), güven eşiği 0,20.  
**Mimariler (her veri seti için AYNI protokolle):** YOLOv8n/s/m/l (ölçek ablasyonu, s=referans) + YOLOv11s (güncel sürüm karşılaştırması).  
**Metrik:** mAP@50, mAP@50-95, Kesinlik, Duyarlılık, F1, sınıf bazlı AP, FPS, Params(M), GFLOPs.

**ÖNEMLİ:** Üst menüden `Çalışma Zamanı` → `Çalışma zamanı türünü değiştir` → **T4 GPU** seçin.

## 1. Kütüphaneler

In [ ]:
!pip install -q ultralytics roboflow

## 2. Veri setlerini indir (Roboflow)

In [ ]:
import os
from roboflow import Roboflow

DATASETS = {
    'catlak': dict(api_key='VjVedHy8GcB2HOWgjUAs', workspace='akash-roy-edivz', project='building-crack-detection-ys4zf', version=3, title='Catlak Tespiti'),
    'bina': dict(api_key='VjVedHy8GcB2HOWgjUAs', workspace='ofgg', project='maras-tpx5p', version=4, title='Bina Hasari Tespiti'),
}

DATA_YAML = {}
for key, d in DATASETS.items():
    print(f'--- {d["title"]} indiriliyor ---')
    rf = Roboflow(api_key=d['api_key'])
    proj = rf.workspace(d['workspace']).project(d['project'])
    ds = proj.version(d['version']).download('yolov8')
    DATA_YAML[key] = f'{ds.location}/data.yaml'
    print(key, '->', DATA_YAML[key])

## 3. Ortak eğitim + değerlendirme fonksiyonu
Bütün mimariler ve her iki veri seti bu AYNI fonksiyonu kullanır.

In [ ]:
import json, time
import torch
from ultralytics import YOLO

RESULTS_DIR = 'results/'
os.makedirs(RESULTS_DIR, exist_ok=True)

EPOCHS, IMG_SIZE, BATCH, CONF = 60, 640, 16, 0.20

def efficiency(model, imgsz=640, runs=30):
    dev = 'cuda' if torch.cuda.is_available() else 'cpu'
    info = model.info(verbose=False) if hasattr(model, 'info') else None
    params_m = (info[1] / 1e6) if info else float('nan')
    gflops = info[3] if info and len(info) > 3 else float('nan')
    x = torch.zeros(1, 3, imgsz, imgsz)
    try:
        pm = model.model.to(dev).eval()
        with torch.no_grad():
            for _ in range(5): pm(x.to(dev))
            if dev == 'cuda': torch.cuda.synchronize()
            t0 = time.time()
            for _ in range(runs): pm(x.to(dev))
            if dev == 'cuda': torch.cuda.synchronize()
        ms = (time.time() - t0) / runs * 1000
    except Exception:
        ms = float('nan')
    return {'params_m': params_m, 'gflops': gflops, 'infer_ms': ms,
            'fps': (1000/ms) if ms == ms else float('nan')}

def train_eval(dataset_key, arch_name, weights):
    data_yaml = DATA_YAML[dataset_key]
    run_name = f'{dataset_key}_{arch_name}'
    model = YOLO(weights)
    model.train(data=data_yaml, epochs=EPOCHS, imgsz=IMG_SIZE, batch=BATCH,
                seed=42, name=run_name, exist_ok=True, verbose=False)
    m = model.val(data=data_yaml, split='test', imgsz=IMG_SIZE, conf=CONF, verbose=False)
    box = m.box
    prec, rec = float(box.mp), float(box.mr)
    metrics = {
        'dataset': dataset_key, 'arch': arch_name,
        'map50': float(box.map50), 'map50_95': float(box.map),
        'precision': prec, 'recall': rec,
        'f1': float(2*prec*rec/(prec+rec+1e-9)),
        'per_class_ap50': {m.names[i]: float(v) for i, v in enumerate(box.ap50)} if hasattr(box, 'ap50') else {},
    }
    metrics.update(efficiency(model, IMG_SIZE))
    json.dump(metrics, open(f'{RESULTS_DIR}{run_name}.json', 'w'), indent=2, ensure_ascii=False)
    map50, map5095, f1, fps = metrics['map50'], metrics['map50_95'], metrics['f1'], metrics['fps']
    print(f'[{run_name}] mAP50={map50:.4f} mAP50-95={map5095:.4f} F1={f1:.4f} FPS={fps:.1f}')
    return metrics

## 4. Bütün veri setleri × mimariler için eğit

In [ ]:
ARCH_WEIGHTS = {'yolov8n': 'yolov8n.pt', 'yolov8s': 'yolov8s.pt', 'yolov8m': 'yolov8m.pt', 'yolov8l': 'yolov8l.pt', 'yolo11s': 'yolo11s.pt'}

all_results = {}
for dataset_key in DATASETS:
    for arch_name, weights in ARCH_WEIGHTS.items():
        key = f'{dataset_key}_{arch_name}'
        print(f"\n{'='*70}\n{key}\n{'='*70}")
        try:
            all_results[key] = train_eval(dataset_key, arch_name, weights)
        except Exception as e:
            print(f'HATA [{key}]: {e}')

## 5. Karşılaştırma tabloları + grafikler (veri seti başına)

In [ ]:
import pandas as pd, matplotlib.pyplot as plt

for dataset_key, d in DATASETS.items():
    rows = [v for k, v in all_results.items() if v.get('dataset') == dataset_key]
    if not rows: continue
    tbl = pd.DataFrame(rows)[['arch','map50','map50_95','precision','recall','f1','params_m','gflops','fps']]
    tbl = tbl.sort_values('map50_95', ascending=False)
    tbl.to_csv(f'{RESULTS_DIR}00_{dataset_key}_comparison.csv', index=False)
    display(tbl.round(4))

    fig, ax = plt.subplots(figsize=(7,4))
    ax.barh(tbl['arch'], tbl['map50_95'], color='#4C72B0')
    ax.set_xlabel('mAP@50-95'); ax.set_title(f"{d['title']} - Mimari Karsilastirmasi")
    plt.tight_layout(); plt.savefig(f'{RESULTS_DIR}00_{dataset_key}_chart.png', dpi=160); plt.show()

## 6. Sonuçları indir

In [ ]:
import shutil
shutil.make_archive('03_yer_seviyesi_sonuclar', 'zip', RESULTS_DIR)
from google.colab import files
files.download('03_yer_seviyesi_sonuclar.zip')